# Chapter 8 · AI-Assisted Ingestion, Profiling, and Documentation

**Book:** *AI-Based Data Engineering* (Packt) | **Case Study:** OpsPulse

---

## Turn Ingestion into a Documentation Engine

Most data ingestion pipelines deliver data—and stop there. The table lands, the pipeline ticks green, and documenting what the data means gets deferred indefinitely.

Chapter 8 shows how to close that loop. Every ingestion run becomes a documentation run: Snowflake Cortex AI functions profile the schema, classify columns, and write business-ready descriptions into the table’s own `COMMENT` fields—all before the analyst opens the table.

The three Cortex capabilities that drive the pipeline:

| Function | What it does |
|---|---|
| `AI_PARSE_DOCUMENT` | Extracts structured text and layout from PDF files on a Snowflake stage |
| `AI_COMPLETE` | Calls a Cortex-hosted LLM inline in SQL—no external API call needed |
| `INFER_SCHEMA` | Detects column names and types from staged CSV/JSON/Parquet files |

This notebook walks through the full pipeline on OpsPulse device-calibration data, from raw staged files to a fully documented Snowflake table.


## Prerequisites

Before running this notebook:

1. **OpsPulse dataset loaded** — run `code/setup/opspulse_generator.py --target snowflake` to populate `OPSPU.RAW`, `OPSPU.MARTS`, and related schemas.
2. **Cortex AI enabled** — `AI_COMPLETE` uses Cortex-hosted models; no external API key is required in SQL cells. Verify Cortex AI is enabled for your Snowflake account.
3. **Sufficient privileges** — `CREATE STAGE` and `ALTER TABLE … MODIFY COLUMN COMMENT` require at minimum `USAGE` on the database/schema plus `CREATE` privileges.

> **OpsPulse IoT context:** The `device_calibration` raw table records every sensor calibration event across all field devices. The ~4% null-timestamp anomaly documented in Chapter 1 originates here—the schema profiler in this notebook surfaces it.


In [ ]:
%%sql -r stage_result
-- Create a stage to hold OpsPulse calibration documents and CSV files.
-- DIRECTORY = (ENABLE = TRUE) allows listing files via DIRECTORY(@stage).
CREATE STAGE IF NOT EXISTS OPSPU.PUBLIC.opspu_calibration_stage
  DIRECTORY = (ENABLE = TRUE);


## Part 1 · AI_PARSE_DOCUMENT—Extract Text from PDF Calibration Reports

`AI_PARSE_DOCUMENT` is a Snowflake Cortex function that reads PDF, Word, and image files directly from a Snowflake stage and returns the extracted text as a `VARIANT`. The `'LAYOUT'` mode preserves spatial structure—headers, tables, and numbered lists come back in reading order.

**Correct syntax (post bug-fix C8-1):**

```sql
AI_PARSE_DOCUMENT(
    TO_FILE(@stage, relative_path),
    {'mode': 'LAYOUT'}
)
```

The older `SNOWFLAKE.CORTEX.PARSE_DOCUMENT(@stage, path)` signature is deprecated and will error at runtime. Always use the `TO_FILE()` wrapper.

> **Note:** `AI_PARSE_DOCUMENT` is scheduled for deprecation end-2026. For new pipelines, prefer `AI_COMPLETE` with multimodal input or Cortex document extraction endpoints as they stabilise.

OpsPulse field teams upload PDF calibration reports to `opspu_calibration_stage`. Step 5 extracts structured text from every PDF in the stage in a single SQL query.


In [ ]:
%%sql -r parse_doc_result
-- AI_PARSE_DOCUMENT extracts text and layout from PDF files on a stage.
-- When you upload PDFs to opspu_calibration_stage, this query processes all of them.
-- If the stage is empty, the SELECT returns 0 rows (not an error).
--
-- To test with a real file: upload any PDF to the stage, then run this query.
SELECT
    relative_path                   AS file_name,
    AI_PARSE_DOCUMENT(
        TO_FILE('@OPSPU.PUBLIC.opspu_calibration_stage', relative_path),
        {'mode': 'LAYOUT'}
    )::VARIANT                      AS document_content
FROM DIRECTORY('@OPSPU.PUBLIC.opspu_calibration_stage')
WHERE relative_path LIKE '%.pdf'
LIMIT 10;


In [ ]:
%%sql -r firmware_parsed
-- AI_COMPLETE calls a Cortex-hosted LLM inline in SQL.
-- Here we extract structured fields from free-text firmware notification emails.
--
-- Source: OPSPU.PUBLIC.FIRMWARE_NOTIFICATIONS (if loaded).
-- Fallback: VALUES CTE below provides three sample notifications as a demo.
--
-- Model: claude-haiku-4-5 -- fast and cost-efficient for structured extraction.
SELECT
    notification_id,
    AI_COMPLETE(
        'claude-haiku-4-5',
        CONCAT(
            'Parse this firmware notification email and return JSON with fields: ',
            'device_model (string), firmware_version (string, e.g. "v4.2.1"), ',
            'severity (one of: critical, major, minor, informational). ',
            'Return only valid JSON, no explanation. Email: ',
            email_body
        )
    )::VARIANT                      AS parsed_firmware_update
FROM (
    VALUES
        (1, 'URGENT: Device model XR-500 firmware v4.2.1 critical security patch required. All XR-500 units must update within 48 hours.'),
        (2, 'Routine: SmartSensor Pro v2.1.3 minor patch available. Optional update. Fixes intermittent Bluetooth pairing issue.'),
        (3, 'IMPORTANT: FieldHub Gateway v3.0.0 major update. Resolves data-loss bug in offline mode. Affects units in regions EU-W and APAC-S.')
) AS t(notification_id, email_body);


## Part 2 · Schema Profiling

Before loading data, profiling the target table tells you what is already there—column types, null rates, cardinality, and min/max ranges. This is the *profile-before-ingest* pattern from Chapter 8 §8.1.

The profiler in `schema_profiling.py` builds a single dynamic SQL query that captures all of these signals in one pass. In this notebook we start with `INFORMATION_SCHEMA.COLUMNS` to inspect the schema, then use `AI_COMPLETE` to generate business-friendly column descriptions.

**Why this matters for OpsPulse:** The `FCT_ACTIVE_CUSTOMERS` mart table is the canonical fact table for active-customer metrics. Documenting its columns directly in Snowflake means any tool that reads `INFORMATION_SCHEMA.COLUMNS.COMMENT`—dbt docs, Cortex Analyst, the Snowflake UI—gets accurate, AI-generated descriptions automatically.


In [ ]:
%%sql -r schema_cols
-- Inspect the FCT_ACTIVE_CUSTOMERS mart table schema.
-- Returns column names, types, nullability, and any existing comments.
SELECT
    column_name,
    data_type,
    is_nullable,
    comment
FROM OPSPU.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'MARTS'
  AND table_name   = 'FCT_ACTIVE_CUSTOMERS'
ORDER BY ordinal_position;


In [ ]:
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

# ── Step 1: pull the schema into a pandas DataFrame ──────────────────────
schema_df = session.sql("""
    SELECT column_name, data_type, is_nullable
    FROM OPSPU.INFORMATION_SCHEMA.COLUMNS
    WHERE table_schema = 'MARTS' AND table_name = 'FCT_ACTIVE_CUSTOMERS'
    ORDER BY ordinal_position
""").to_pandas()

print("=== FCT_ACTIVE_CUSTOMERS Schema ===")
print(schema_df.to_string(index=False))
print(f"\n{len(schema_df)} columns found.")

# ── Step 2: preview the generator prompts ────────────────────────────────
# The evaluator-optimizer loop in documentation_pipeline.py calls AI_COMPLETE
# twice per column: GENERATE (claude-haiku-4-5) then EVALUATE (claude-sonnet-4-5).
# If the evaluator score < 0.85, the generator retries with feedback (max 3 rounds).
# The SQL cell below runs the generator pass for the first 5 columns.

def build_generator_prompt(column_name: str, data_type: str) -> str:
    return (
        f"Write a one-sentence column description for '{column_name}' ({data_type}) "
        f"in the FCT_ACTIVE_CUSTOMERS table. "
        f"Context: OpsPulse is an IoT + CRM platform tracking active customers. "
        f"State the business meaning and key constraints. "
        f"Under 150 characters. No jargon. Return only the description."
    )

# Preview prompts for the first three columns
print("\n=== Generator prompt preview (first 3 columns) ===")
for _, row in schema_df.head(3).iterrows():
    prompt = build_generator_prompt(row["COLUMN_NAME"], row["DATA_TYPE"])
    print(f"\n[{row['COLUMN_NAME']}] ({len(prompt)} chars):")
    print(f"  {prompt}")


In [ ]:
%%sql -r col_descriptions
-- Generator pass: AI_COMPLETE writes a business description for each column.
-- Model: claude-haiku-4-5 -- fast, cost-efficient for short structured-output tasks.
--
-- The evaluator pass (in documentation_pipeline.py) checks each description against:
--   1. Business meaning -- conveys more than the column name alone.
--   2. Jargon-free -- a business analyst understands it without engineering context.
--   3. Under 150 characters -- fits a Snowflake COMMENT field without truncation.
-- If any criterion fails, the generator retries with reviewer feedback (max 3 rounds).
SELECT
    column_name,
    data_type,
    AI_COMPLETE(
        'claude-haiku-4-5',
        CONCAT(
            'Write a one-sentence business description for a column named "',
            column_name, '" (', data_type, ') ',
            'in the FCT_ACTIVE_CUSTOMERS table. ',
            'Context: OpsPulse is an IoT + CRM platform tracking active customers across ERP, CRM, and IoT telemetry sources. ',
            'State the business meaning and key constraints. Under 150 characters. No jargon. ',
            'Return only the description, no preamble.'
        )
    ) AS ai_description
FROM OPSPU.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'MARTS'
  AND table_name   = 'FCT_ACTIVE_CUSTOMERS'
ORDER BY ordinal_position
LIMIT 5;


In [ ]:
%%sql -r apply_preview
-- Apply AI-generated descriptions to the table as column comments.
-- In production, replace the preview SELECT with the ALTER statements.
--
-- Example (uncomment to execute):
-- ALTER TABLE OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS
--   MODIFY COLUMN customer_id
--   COMMENT 'Unique identifier for an active OpsPulse customer across all data sources.';
--
-- Applying comments makes descriptions visible in:
--   INFORMATION_SCHEMA.COLUMNS.COMMENT  /  Snowflake Data Catalog (Horizon)
--   Cortex Analyst semantic context      /  dbt docs (via schema.yml sync)
--
-- Preview: show which columns would receive an AI-generated comment.
SELECT
    column_name,
    data_type,
    COALESCE(comment, '(no comment yet)') AS current_comment,
    'AI-generated description will be applied here' AS planned_action
FROM OPSPU.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'MARTS'
  AND table_name   = 'FCT_ACTIVE_CUSTOMERS'
ORDER BY ordinal_position
LIMIT 3;


## Summary

This notebook demonstrated the documentation-as-byproduct pipeline from Chapter 8:

| Step | Cortex capability | What it produced |
|---|---|---|
| Stage creation | — | `opspu_calibration_stage` ready for file uploads |
| Document extraction | `AI_PARSE_DOCUMENT` | Structured text from PDF calibration reports |
| Email parsing | `AI_COMPLETE` | Structured JSON fields from free-text firmware notifications |
| Schema inspection | `INFORMATION_SCHEMA.COLUMNS` | Column names, types, and nullability |
| Column documentation | `AI_COMPLETE` | Business-ready descriptions for every column |
| Documentation preview | `ALTER TABLE … MODIFY COLUMN COMMENT` | Comments ready to write back to the table |

**OpsPulse outcome:** `FCT_ACTIVE_CUSTOMERS` goes from zero column comments to AI-generated, evaluator-verified descriptions in a single pipeline run—with no human writing involved.

**Next in Chapter 8:** The full `documentation_pipeline.py` adds two more steps—an OpenLineage `RunEvent` emission and a dbt `schema.yml` block generator—so every ingestion run also updates your data catalog and dbt project documentation automatically.

**Elapsed time (from book §8.5):** 4 minutes pipeline runtime, 10 minutes engineer review. Manual equivalent: 2–3 hours.
